# Fields to update
1. MB_ID (Done)
2. Country
3. Sub_genre
4. Homepage
5. Bandcamp page

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [ ]:
# Notion imports
from Notion.reader import get_artists_db
from Notion.database import myNotionDatabases

# MusicBrainz imports
from Datasources.MusicBrainz import search_artist,get_artist_info

import random

In [ ]:
notion_response = get_artists_db()
artists_db = []
db = myNotionDatabases()
for artist in notion_response:
    artists_db.append(db.extract_artist(artist))

### Artists producing errors:
* Senses - Error - Exists under MB ID "9941e53e-601b-4fc3-9d08-6bc770c4c65c" I suspect its a confidence thing.
    * Not even a confidence thing, there are too many artists called the same thing. MB ID doesnt know what to do here.
    * Need to handle this better, ignore exact matchs if the results have more than 1 exact match.
    * Even after doing this its wrong, its returning "Senses Fail" totaly different band.
* Warduna - No match - Does not show up on MB at all.
    * Fine this will be manual.
* Hanabie - No Match - Seems to be an issue with the fact they are from japan. 
    * Might need to look for their sort name
    * Will keep manual for now.
* Windrose - No match
    * MB has them under "592b488d-dd42-4d9a-a343-d7ca4f69d0bb" but the name is "wind rose". Need to check what it should be from the bands sites.

In [ ]:
#for artist in random.sample(artists_db, min(10, len(artists_db))): #Selecting 5 random artists for testing purposes
for artist in artists_db: #Iterating through all artists in the database
    if artist["mb_id"] is None:
        try:
            mb_id, match_result = search_artist(artist_name=artist["artist_name"], confidence_threshold=85)
            if mb_id is None:
                print(f"No match found for {artist['artist_name']} with the reason: {match_result}")
                continue
            
            mb_artist_info = get_artist_info(mb_id)
            print(f'{artist["artist_name"]} - {mb_id} - {match_result} - {mb_artist_info["country"].name} - {mb_artist_info["homepage"]} - {mb_artist_info["bandcamp"]} ')
        except Exception as e:
            print(f"Error processing {artist['artist_name']} - {mb_id}: {e}")

In [ ]:
#Get the Last.fm info for the artist.

In [ ]:
#Commenting out for now to avoid damaging database with test runs. Uncomment to update mb_id in Notion.
#update_mb_id(artist["notion_id"], mb_id)